### Global Forest Recovery: Do Forest Carbon Storage changes corelate to  suggest recovery?

In [1]:
!pip install plotly country_converter -q

In [2]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import country_converter as coco
import logging

In [3]:
df = pd.read_csv("data/forest_and_carbon_data.csv")

## Proposition: Forest Loss Continues Despite Recovery Efforts

## Honest Interpretation
### Percent Change in Forest Extent (1992–2022)

In [72]:
import json
from plotly.subplots import make_subplots
import plotly.graph_objects as go

logging.getLogger("country_converter").setLevel(logging.ERROR)
pio.renderers.default = "iframe_connected"

forest_df = df[df["INDICATOR"] == "Forest area, 1000 Hectares"].copy()
forest_df = forest_df.set_index("COUNTRY")
forest_df["1992"] = pd.to_numeric(forest_df["1992"], errors="coerce")
forest_df["2023"] = pd.to_numeric(forest_df["2022"], errors="coerce")

pct_change = ((forest_df["2022"] - forest_df["1992"]) / forest_df["1992"] * 100).dropna()
pct_change = pct_change.reset_index()
pct_change.columns = ["COUNTRY", "pct_change"]

pct_change["iso3"] = coco.convert(pct_change["COUNTRY"], to="ISO3", not_found=None)
pct_change = pct_change[pct_change["iso3"] != "not found"].dropna(subset=["iso3"])
pct_change = pct_change[pct_change["iso3"].apply(lambda x: isinstance(x, str))]

print(f"Countries mapped: {len(pct_change)}")

bins = [-float("inf"), -50, -20, -5, 5, 20, 50, float("inf")]
labels = [
    "Deep Loss (≤ -50%)", "Loss (-50% to -20%)", "Slight Loss (-20% to -5%)",
    "Stable (-5% to 5%)", "Slight Gain (5% to 20%)", "Gain (20% to 50%)", "Large Gain (≥ 50%)"
]

pct_change["category"] = pd.cut(pct_change["pct_change"], bins=bins, labels=labels)

# Global stats
_f1992  = pd.to_numeric(forest_df["1992"], errors="coerce")
_f2022  = pd.to_numeric(forest_df["2022"], errors="coerce")
_in_map = forest_df.index.isin(pct_change["COUNTRY"])
_g1992  = float(_f1992[_in_map].sum())
_g2022  = float(_f2022[_in_map].sum())
global_pct_val = round((_g2022 - _g1992) / _g1992 * 100, 2)
global_net_kha = round(_g2022 - _g1992, 0)

# Global yearly totals for scatter
_years = [str(y) for y in range(1992, 2023)]
_global_yearly = (
    forest_df[_in_map][_years]
    .apply(pd.to_numeric, errors="coerce")
    .sum()
    .reset_index()
)
_global_yearly.columns = ["Year", "ForestArea_kha"]
_global_yearly["Year"] = _global_yearly["Year"].astype(int)
_global_yearly["ForestArea_Mha"] = _global_yearly["ForestArea_kha"] / 1000

# ── Recovery deficit stats ─────────────────────────────────────────────────────
_in_loss = forest_df.index.isin(pct_change.loc[pct_change["pct_change"] < 0, "COUNTRY"])
_in_gain = forest_df.index.isin(pct_change.loc[pct_change["pct_change"] > 0, "COUNTRY"])

_total_loss_mha   = round((_f1992[_in_loss].sum() - _f2022[_in_loss].sum()) / 1000, 1)
_total_gain_mha   = round((_f2022[_in_gain].sum() - _f1992[_in_gain].sum()) / 1000, 1)
_net_deficit_mha  = round(_total_loss_mha - _total_gain_mha, 1)
_mha_per_year     = round(_net_deficit_mha / 4, 1)
_loss_count       = int((pct_change["pct_change"] < 0).sum())
_gain_count       = int((pct_change["pct_change"] > 0).sum())
_annual_gain_rate = round(_total_gain_mha / 30, 1)
_rate_multiplier  = round(_mha_per_year / _annual_gain_rate, 1) if _annual_gain_rate > 0 else 0

deficit_stats = {
    "loss_mha":     _total_loss_mha,
    "gain_mha":     _total_gain_mha,
    "deficit_mha":  _net_deficit_mha,
    "mha_per_year": _mha_per_year,
    "loss_count":   _loss_count,
    "gain_count":   _gain_count,
    "multiplier":   _rate_multiplier,
}
print(f"Lost: {_total_loss_mha} Mha | Gained: {_total_gain_mha} Mha | Deficit: {_net_deficit_mha} Mha | Need: {_mha_per_year} Mha/yr | {_rate_multiplier}x current rate")

color_map = {
    "Deep Loss (≤ -50%)":        "#8c510a",
    "Loss (-50% to -20%)":       "#bf812d",
    "Slight Loss (-20% to -5%)": "#dfc27d",
    "Stable (-5% to 5%)":        "#ebf1e7",
    "Slight Gain (5% to 20%)":   "#80cdc1",
    "Gain (20% to 50%)":         "#35978f",
    "Large Gain (≥ 50%)":        "#01665e",
}

BG      = "#1c1c2e"
TEXT    = "#e8e8f0"
SUBTEXT = "#b0b0c8"
GRID    = "#2e2e48"
CARD_BG = "#252538"

_cho = px.choropleth(
    pct_change,
    locations="iso3",
    color="category",
    hover_name="COUNTRY",
    hover_data={"pct_change": ":.1f", "category": False, "iso3": False},
    category_orders={"category": labels},
    color_discrete_map=color_map,
    labels={"category": "% Change in Forest Area"}
)

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.82, 0.18],
    specs=[[{"type": "choropleth"}], [{"type": "scatter"}]],
    vertical_spacing=0.005,
    subplot_titles=("", "Global Time Series Projection"),
)

for trace in _cho.data:
    fig.add_trace(trace, row=1, col=1)

fig.add_trace(
    go.Scatter(
        x=_global_yearly["Year"],
        y=_global_yearly["ForestArea_Mha"],
        mode="lines+markers",
        line=dict(color="#ebf1e7", width=2.5),
        marker=dict(color="#ebf1e7", size=5),
        hovertemplate="%{x}: %{y:.0f} Mha<extra></extra>",
        showlegend=False,
    ),
    row=2, col=1,
)

fig.update_geos(
    showcoastlines=True,
    showframe=False,
    showcountries=True,
    landcolor="#2e2e48",
    bgcolor=CARD_BG,
    showocean=True,
    oceancolor=CARD_BG,
    showlakes=True,
    lakecolor=CARD_BG,
    fitbounds=False,
)

fig.update_layout(
    width=955,
    height=600,
    paper_bgcolor=BG,
    plot_bgcolor=CARD_BG,
    font=dict(color=TEXT),
    title_text="Rooted in Reality: Restoration Efforts Are Outpaced by Forest Loss <br><sub>Tracking progress of the Glasgow Leaders Declaration to stop and reverse deforestation by 2030 </sub>",
    title_font=dict(size=20, color=TEXT),
    title_x=0.50,
    title_y=0.97,
    legend_title_text="% Change in Forest Area",
    legend=dict(
        x=1.0, y=0.85, xanchor="left", yanchor="middle",
        font=dict(color=TEXT),
        title_font=dict(color=SUBTEXT),
        bgcolor="rgba(28,28,46,0.0)",
        itemclick=False,
        itemdoubleclick=False,
    ),
    margin=dict(l=0, r=200, t=60, b=15),
    xaxis2=dict(
        title="Year", tickmode="linear", dtick=5,
        title_font=dict(color=SUBTEXT),
        tickfont=dict(color=SUBTEXT),
        gridcolor=GRID,
        linecolor=GRID,
        zerolinecolor=GRID,
    ),
    yaxis2=dict(
        title="Forest Area (Mha)",
        title_font=dict(color=SUBTEXT),
        tickfont=dict(color=SUBTEXT),
        gridcolor=GRID,
        linecolor=GRID,
        zerolinecolor=GRID,
    ),
)

# Subplot title color (time series label)
for ann in fig.layout.annotations:
    ann.font = dict(color=SUBTEXT, size=11)

# Subtitle: bottom-left, just below the choropleth map area
fig.add_annotation(
    x=0.0,
    y=0.215,
    xref="paper",
    yref="paper",
    text="Percent change in forest area, 1992–2022  |  Source: International Monetary Fund (IMF)",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(color=SUBTEXT, size=10),
)

post_js = f"""
(function() {{
    var gd = document.getElementsByClassName('plotly-graph-div')[0];
    if (!gd) return;

    document.body.style.background = '{BG}';
    document.body.style.margin = '0';

    var style = document.createElement('style');
    style.textContent = '.legend .traces rect.legendtoggle {{ cursor: default !important; }}';
    document.head.appendChild(style);

    var colorMap    = {json.dumps(color_map)};
    var globalStats = {json.dumps({"pct": global_pct_val, "net_kha": global_net_kha})};
    var deficitStats = {json.dumps(deficit_stats)};

    gd.style.position = 'relative';

    // ── Side stat cards ────────────────────────────────────────────────────────
    var d = deficitStats;
    var sideCardDefs = [
        {{ label: 'Forest Lost',    value: d.loss_mha.toFixed(1) + ' Mha',        sub: d.loss_count + ' countries',  accent: '#8c510a' }},
        {{ label: 'Forest Gained',  value: d.gain_mha.toFixed(1) + ' Mha',        sub: d.gain_count + ' countries',  accent: '#35978f' }},
        {{ label: 'Net Deficit',    value: d.deficit_mha.toFixed(1) + ' Mha',     sub: 'unrecovered loss',            accent: '#ebf1e7' }},
        {{ label: 'Needed by 2030', value: d.mha_per_year.toFixed(1) + ' Mha/yr', sub: 'to break even',              accent: '#f97316' }},
    ];

    var sideCards = document.createElement('div');
    sideCards.style.cssText = 'position:absolute;left:762px;top:230px;width:183px;font-family:Arial,sans-serif;pointer-events:none';
    sideCards.innerHTML = sideCardDefs.map(function(s) {{
        return '<div style="border-left:3px solid ' + s.accent + ';padding:6px 10px;margin-bottom:7px;background:#252538;border-radius:0 3px 3px 0">' +
            '<div style="font-size:8px;letter-spacing:1.4px;text-transform:uppercase;color:' + s.accent + ';margin-bottom:2px">' + s.label + '</div>' +
            '<div style="font-size:16px;font-weight:bold;color:#e8e8f0;line-height:1.1;margin-bottom:1px">' + s.value + '</div>' +
            '<div style="font-size:10px;color:#8080a0">' + s.sub + '</div>' +
            '</div>';
    }}).join('');
    gd.appendChild(sideCards);

    // ── Recovery deficit callout ───────────────────────────────────────────────
    var narrative =
        'Net-loss countries have lost <strong style="color:#fff">' + d.loss_mha.toFixed(1) + ' Mha</strong> of forest since 1992 ' +
        'across <strong style="color:#fff">' + d.loss_count + ' countries</strong>. ' +
        'Gains across <strong style="color:#fff">' + d.gain_count + ' countries</strong> have recovered only ' +
        '<strong style="color:#fff">' + d.gain_mha.toFixed(1) + ' Mha</strong>, leaving a deficit of ' +
        '<strong style="color:#f97316">' + d.deficit_mha.toFixed(1) + ' Mha</strong>. ' +
        'Closing that gap by 2030 requires <strong style="color:#f97316">' + d.mha_per_year.toFixed(1) + ' Mha of restoration per year</strong>, ' +
        '<strong style="color:#f97316">' + d.multiplier.toFixed(1) + '× the annual rate achieved over the last 30 years.</strong>';

    var callout = document.createElement('div');
    callout.style.cssText = 'width:955px;background:#252538;color:#e8e8f0;border-radius:8px;padding:20px 28px 22px;margin-top:4px;font-family:Arial,sans-serif;box-sizing:border-box;border:1px solid #2e2e48';
    callout.innerHTML =
        '<div style="font-size:10px;letter-spacing:2px;text-transform:uppercase;color:#f97316;margin-bottom:10px">The Recovery Deficit</div>' +
        '<div style="font-size:13px;color:#b0b0c8;line-height:1.7">' + narrative + '</div>';
    gd.insertAdjacentElement('afterend', callout);

    // ── Stats box (global summary, static) ────────────────────────────────────
    var statsBox = document.createElement('div');
    statsBox.style.cssText = [
        'position:absolute',
        'left:762px',
        'top:494px',
        'width:160px',
        'background:#252538',
        'border:1px solid #2e2e48',
        'border-radius:5px',
        'padding:10px 12px',
        'font-family:Arial,sans-serif',
        'font-size:12px',
        'line-height:1.6',
        'color:#e8e8f0',
        'box-shadow:0 1px 5px rgba(0,0,0,0.4)',
        'pointer-events:none',
    ].join(';');

    var pct = globalStats.pct;
    var c;
    if      (pct <= -50) c = colorMap['Deep Loss (≤ -50%)'];
    else if (pct <= -20) c = colorMap['Loss (-50% to -20%)'];
    else if (pct <=  -5) c = colorMap['Slight Loss (-20% to -5%)'];
    else if (pct <=   5) c = colorMap['Stable (-5% to 5%)'];
    else if (pct <=  20) c = colorMap['Slight Gain (5% to 20%)'];
    else if (pct <=  50) c = colorMap['Gain (20% to 50%)'];
    else                  c = colorMap['Large Gain (≥ 50%)'];
    var pctColor = (c === '#ebf1e7' || c === '#ffffff') ? '#b0b0c8' : c;

    var sign    = pct >= 0 ? '+' : '';
    var netMha  = globalStats.net_kha / 1000;
    var netSign = netMha >= 0 ? '+' : '';
    statsBox.innerHTML =
        '<div style="font-weight:bold;font-size:13px;border-bottom:1px solid #2e2e48;' +
        'padding-bottom:5px;margin-bottom:6px;color:#e8e8f0">Global (1992–2022)</div>' +
        '<div style="margin-bottom:3px;color:#b0b0c8">Change: <strong style="color:' + pctColor + '">' +
        sign + pct.toFixed(1) + '%</strong></div>' +
        '<div style="color:#b0b0c8">Net: <strong style="color:' + pctColor + '">' +
        netSign + netMha.toFixed(1) + ' Mha</strong></div>';
    gd.appendChild(statsBox);
}})();
"""

fig.write_html("forest_loss_map.html", post_script=post_js)
print("Saved: forest_loss_map.html")
fig.show()


Countries mapped: 230
Lost: 799.0 Mha | Gained: 256.4 Mha | Deficit: 542.6 Mha | Need: 135.6 Mha/yr | 16.0x current rate
Saved: forest_loss_map.html


In [170]:
total_1992 = forest_df["1992"].sum()
total_2022 = forest_df["2022"].sum()

pct_global = (total_2022 - total_1992) / total_1992 * 100

print("Global Forest Area:", pct_global)

Global Forest Area: -2.5479275043177023


In [171]:
net_change = (forest_df["2022"] - forest_df["1992"]).sum()

In [172]:
print("Net Change in Forest Area:", net_change)

Net Change in Forest Area: -542575.5896015903


## Dishonest Interpretation
### Percent Change in Forest Extent (1992–2022)

In [92]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import country_converter as coco
import logging, json

logging.getLogger("country_converter").setLevel(logging.ERROR)
pio.renderers.default = "iframe_connected"

# ── Data prep ────────────────────────────────────────────────────────────────
_exclude = [
    "G7", "G20", "Asia", "Africa", "Americas",
    "Euro Area (EA) - aggregate of member states", "World",
    "European Union", "Advanced Economies",
    "Emerging Market and Developing Economies",
]
fdf = df[df["INDICATOR"] == "Forest area, 1000 Hectares"].copy()
fdf = fdf.set_index("COUNTRY")
fdf = fdf[~fdf.index.isin(_exclude)]
_years = [str(y) for y in range(1992, 2023)]
for y in _years:
    fdf[y] = pd.to_numeric(fdf[y], errors="coerce")

# Choropleth: 1992→2022 pct change, 3 bins
pct_change = ((fdf["2022"] - fdf["1992"]) / fdf["1992"] * 100).dropna().reset_index()
pct_change.columns = ["COUNTRY", "pct_change"]
pct_change["iso3"] = coco.convert(pct_change["COUNTRY"], to="ISO3", not_found=None)
pct_change = pct_change[(pct_change["iso3"] != "not found") & pct_change["iso3"].notna()]

bins, bin_labels = [-float("inf"), -0.1, 0.1, float("inf")], ["Loss", "Stable", "Gain"]
pct_change["category"] = pd.cut(pct_change["pct_change"], bins=bins, labels=bin_labels)

_gain   = int((pct_change["category"] == "Gain").sum())
_loss   = int((pct_change["category"] == "Loss").sum())
_stable = int((pct_change["category"] == "Stable").sum())
_sg     = _gain + _stable
_total  = len(pct_change)
print(f"Loss:{_loss}  Stable:{_stable}  Gain:{_gain}  Stable+Gain:{_sg}  Total:{_total}")

# Cumulative baseline: for each year t, count countries above their 1992 level
base = fdf["1992"]
cum_rows = []
for cy in _years[1:]:
    valid = base.notna() & fdf[cy].notna() & (base > 0)
    pct   = (fdf[cy][valid] - base[valid]) / base[valid] * 100
    cum_rows.append({"Year": int(cy), "Gaining": int((pct > 0.1).sum())})

trend = pd.DataFrame(cum_rows)

_gain_start = int(trend.iloc[0]["Gaining"])
_gain_end   = int(trend.iloc[-1]["Gaining"])
print(f"Countries above 1992 baseline: {_gain_start} (1993) → {_gain_end} (2022)")

# ── Colors ───────────────────────────────────────────────────────────────────
BG, TEXT, SUBTEXT, GRID, CARD = "#1c1c2e", "#e8e8f0", "#b0b0c8", "#2e2e48", "#252538"
color_map = {"Loss": "#efe6d7", "Stable": "#ffffff", "Gain": "#00b140"}

# ── Choropleth traces from px ────────────────────────────────────────────────
_cho = px.choropleth(
    pct_change, locations="iso3", color="category",
    hover_name="COUNTRY",
    hover_data={"pct_change": ":.1f", "category": False, "iso3": False},
    category_orders={"category": bin_labels},
    color_discrete_map=color_map,
    labels={"category": "Forest Status"},
).data

# ── Figure ───────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.81, 0.19],
    specs=[[{"type": "choropleth"}], [{"type": "scatter"}]],
    vertical_spacing=0.06,
    subplot_titles=("", "Countries With Net Forest Gain Relative to 1992 Baseline"),
)

for t in _cho:
    fig.add_trace(t, row=1, col=1)

# Raw gaining line (sole line)
fig.add_trace(go.Scatter(
    x=trend["Year"], y=trend["Gaining"],
    mode="lines", line=dict(color="#00b140", width=2),
    showlegend=False,
    hovertemplate="%{x}: %{y}<extra>Gaining</extra>",
), row=2, col=1)

# ── Geo / layout ─────────────────────────────────────────────────────────────
fig.update_geos(
    showcoastlines=True, showframe=False, showcountries=True,
    landcolor="#d0d0d0", row=1, col=1,
)

fig.update_layout(
    width=955, height=650,
    paper_bgcolor=BG, plot_bgcolor=CARD,
    font=dict(color=TEXT),
    title_text=(
        "Global Forest Recovery is Underway"
        "<br><sup>Glasgow Leaders' Declaration (COP26, 2021): "
        "An agreement between 100+ countries to stop and reverse deforestation by 2030</sup>"
    ),
    title_font=dict(size=20, color=TEXT),
    title_x=0.50, title_y=0.98,
    showlegend=True,
    legend=dict(
        x=1.05, y=0.85, xanchor="left", yanchor="middle",
        font=dict(color=TEXT),
        title_text="Forest Status",
        title_font=dict(color=SUBTEXT),
        bgcolor="rgba(37,37,56,0.85)",
        bordercolor=GRID,
        borderwidth=1,
        itemclick=False,
        itemdoubleclick=False,
    ),
    margin=dict(l=30, r=180, t=65, b=15),
    xaxis2=dict(
        title="Year", tickmode="linear", dtick=5,
        title_font=dict(color=SUBTEXT), tickfont=dict(color=SUBTEXT),
        gridcolor=GRID, linecolor=GRID, zerolinecolor=GRID,
    ),
    yaxis2=dict(
        title="# Countries",
        title_font=dict(color=SUBTEXT), tickfont=dict(color=SUBTEXT),
        gridcolor=GRID, linecolor=GRID, zerolinecolor=GRID,
    ),
)

for ann in fig.layout.annotations:
    ann.font = dict(color=SUBTEXT, size=11)

# ── Post-script: side callout cards ──────────────────────────────────────────
_side = {"sg": _sg, "loss": _loss, "gain_start": _gain_start, "gain_end": _gain_end}

post_js = f"""
(function() {{
    var gd = document.getElementsByClassName('plotly-graph-div')[0];
    if (!gd) return;
    document.body.style.background = '{BG}';
    document.body.style.margin = '0';

    var s = {json.dumps(_side)};
    var cards = [
        {{
            label:  'Stable or Gaining',
            value:  s.sg + ' countries',
            sub:    'net change since 1992',
            accent: '#00b140'
        }},
        {{
            label:  'More Gaining Since 1993',
            value:  '+' + (s.gain_end - s.gain_start),
            sub:    s.gain_start + ' → ' + s.gain_end + ' countries',
            accent: '#00b140'
        }},
        {{
            label:  'Net Losing',
            value:  s.loss + ' countries',
            sub:    'net change since 1992',
            accent: '#bf7a3a'
        }},
    ];

    var sidebar = document.createElement('div');
    sidebar.style.cssText = [
        'position:absolute', 'left:778px', 'top:280px',
        'width:168px', 'font-family:Arial,sans-serif', 'pointer-events:none'
    ].join(';');
    sidebar.innerHTML = cards.map(function(c) {{
        return '<div style="border-left:3px solid ' + c.accent + ';padding:7px 10px;margin-bottom:9px;' +
               'background:#252538;border-radius:0 3px 3px 0">' +
               '<div style="font-size:8px;letter-spacing:1.4px;text-transform:uppercase;' +
               'color:' + c.accent + ';margin-bottom:2px">' + c.label + '</div>' +
               '<div style="font-size:20px;font-weight:bold;color:#e8e8f0;line-height:1.1;margin-bottom:1px">' + c.value + '</div>' +
               '<div style="font-size:10px;color:#8080a0">' + c.sub + '</div></div>';
    }}).join('');
    gd.appendChild(sidebar);
}})();
"""

fig.write_html("forest_gain_map.html", post_script=post_js)
print("Saved: forest_gain_map.html")
fig.show()


Loss:101  Stable:32  Gain:92  Stable+Gain:124  Total:225
Countries above 1992 baseline: 67 (1993) → 92 (2022)
Saved: forest_gain_map.html


In [174]:
pct_change.nlargest(1, "pct_change")

,COUNTRY,pct_change,iso3,category
159,"Bahrain, Kingdom of",196.0,BHR,Gain


## Honest #2

In [178]:
forest_country_df = forest_df[
    ~forest_df.index.isin(exclude)
].copy()

years = [str(y) for y in range(1992, 2023)]

global_forest = (
    forest_country_df[years]
    .apply(pd.to_numeric, errors="coerce")
    .sum()
)

global_forest = global_forest.reset_index()
global_forest.columns = ["Year", "ForestArea"]

global_forest["Year"] = global_forest["Year"].astype(int)

fig = px.line(
    global_forest,
    x="Year",
    y="ForestArea",
    title="Forest Loss Continues Despite Recovery Efforts"
)

fig.update_traces(
    line=dict(
        color="#8c510a",
        width=4
    )
)

fig.update_layout(
    width=1000,
    height=600,
    xaxis_title="Year",
    yaxis_title="Forest Area (1000 Hectares)",
    showlegend=False
)
fig.write_html("forest_reduction_line.html")
print("Saved: forest_reduction_line.html")
fig.show()

Saved: forest_reduction_line.html


## Dishonest #2
#### Forest recovery efforts have produced substantial gains in a number of countries, resulting in a long positive tail in the distribution.

In [74]:
forest_df["change_pct"] = (
    (forest_df["2022"] - forest_df["1992"])
    / forest_df["1992"]
    * 100
)

all_df = (
    forest_df[["change_pct"]]
    .dropna()
    .reset_index()
    .sort_values("change_pct", ascending=False)
)

all_df["iso3"] = coco.convert(
    all_df["COUNTRY"],
    to="ISO3",
    not_found=None
)

all_df = all_df[
    (all_df["iso3"].notna()) &
    (all_df["iso3"] != "not found")
]

exclude = [
    "G7",
    "G20",
    "Asia",
    "Africa",
    "Americas",
    "Euro Area (EA) - aggregate of member states",
    "World",
    "European Union",
    "Advanced Economies",
    "Emerging Market and Developing Economies"
]

all_df = all_df[
    ~all_df["COUNTRY"].isin(exclude)
]

In [75]:
fig = px.histogram(
    all_df,
    x="change_pct",
    nbins=40,
    title="Forest Gain By Country: Distribution of Forest Area Change by Country"
)

fig.add_vline(x=0, line_dash="dash")
fig.write_html("forest_gain_dist.html")
print("Saved: forest_gain_dist.html")
fig.show()

Saved: forest_gain_dist.html
